# Lab 2: Create a Simple AI Agent

In this lab, we'll introduce you to AI agents by creating a simple agent that will create a bar graph based on data that we give to it.

#### Step 1: Load packages

In [1]:
import os
from typing import Any
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.agents.models import CodeInterpreterTool # previously in azure.ai.projects.models

load_dotenv() # Load environment variables from .env file

True

#### Step 2: Connect to your Microsoft Foundry project

In [2]:
# Connect to Microsoft Foundry project using DefaultAzureCredential, a type of token-based authentication.
project = AIProjectClient(
    endpoint=os.getenv("AIPROJECT_ENDPOINT"),
    credential=DefaultAzureCredential()
)

#### Step 3: Create the simple AI Agent

This step demonstrates how to use the Azure AI Agents SDK to create and interact with an AI agent that can interpret code. The process includes:
- Initializing a code interpreter tool and creating an agent with it.
- Creating a communication thread for the agent.
- Sending a message to the agent with a request to generate a bar chart from provided health plan data.
- Running the agent and monitoring the run status.
- Retrieving and displaying all messages in the thread, and downloading any generated image file.
- Cleaning up by deleting the agent after the process is complete.

This workflow shows how to automate data analysis and visualization tasks using conversational AI agents in Azure.

In [ ]:
import time

def _read_field(obj, name, default=None):
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)

code_interpreter = CodeInterpreterTool()
project = AIProjectClient(
    endpoint=os.getenv("AIPROJECT_ENDPOINT"),
    credential=DefaultAzureCredential()
)

with project:
    # Create an agent with the CodeInterpreterTool
    agent = project.agents.create_agent(
        model=os.environ["CHAT_MODEL"],
        name="my-agent",  # Name of the agent
        instructions="You are a helpful agent",  # Instructions for the agent
        tools=code_interpreter.definitions,
    )
    print(f"Created agent, ID: {agent.id}")

    thread = None
    found_file = False
    try:
        # Create a thread for communication
        thread = project.agents.threads.create()
        print(f"Created thread, ID: {thread.id}")
        
        # Add a message to the thread
        message = project.agents.messages.create(
            thread_id=thread.id,
            role="user",
            content=(
                "Could you please create a bar chart for the using the following data and provide the file to me? "
                "Name the file as health-plan-comparision.png.\n\n"
                "Here is the data:\n"
                "Provider        Monthly Premium    Deductible    Out-of-Pocket Limit\n"
                "Northwind       $300               $1,500        $6,000\n"
                "Aetna           $350               $1,000        $5,500\n"
                "United Health   $250               $2,000        $7,000\n"
                "Premera         $200               $2,200        $6,500\n"
            ),
        )
        print(f"Created message, ID: {message['id']}")
        
        # Create and process an agent run
        run = project.agents.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)
        print(f"Run finished with status: {run.status}")
        
        # Check if the run failed
        if run.status == "failed":
            print(f"Run failed: {run.last_error}")
        
        # Fetch and log all messages
        messages = project.agents.messages.list(thread_id=thread.id)
        print("Conversation:")

        file_targets = []
        for msg in messages:
            print(f"{msg.role}: {msg.content}")
            content_items = _read_field(msg, "content", []) or []

            for item in content_items:
                item_type = _read_field(item, "type", "")

                if item_type == "image_file":
                    image_file = _read_field(item, "image_file", {})
                    file_id = _read_field(image_file, "file_id")
                    if file_id:
                        file_targets.append((file_id, "health-plan-comparision.png"))

                if item_type == "text":
                    text_obj = _read_field(item, "text", {})
                    annotations = _read_field(text_obj, "annotations", []) or []
                    for ann in annotations:
                        ann_type = _read_field(ann, "type", "")
                        ann_text = _read_field(ann, "text", "generated-file.bin")

                        if ann_type == "file_path":
                            file_path = _read_field(ann, "file_path", {})
                            file_id = _read_field(file_path, "file_id")
                            file_name = Path(ann_text).name or "generated-file.bin"
                            if file_id:
                                file_targets.append((file_id, file_name))

                        if ann_type == "file_citation":
                            file_citation = _read_field(ann, "file_citation", {})
                            file_id = _read_field(file_citation, "file_id")
                            file_name = Path(ann_text).name
                            if not file_name or "." not in file_name:
                                file_name = "health-plan-comparision.png"
                            if file_id:
                                file_targets.append((file_id, file_name))

            # Backward compatibility with older SDK message shape
            if hasattr(msg, "file_path_annotations") and msg.file_path_annotations:
                for file_path_annotation in msg.file_path_annotations:
                    file_name = Path(file_path_annotation.text).name
                    file_id = file_path_annotation.file_path.file_id
                    file_targets.append((file_id, file_name))

        # Remove duplicate file ids while preserving order
        deduped_targets = []
        seen_ids = set()
        for file_id, file_name in file_targets:
            if file_id not in seen_ids:
                deduped_targets.append((file_id, file_name))
                seen_ids.add(file_id)

        if not deduped_targets:
            print("No downloadable file references were returned by the agent.")

        for file_id, file_name in deduped_targets:
            download_succeeded = False
            for attempt in range(1, 7):
                try:
                    project.agents.files.save(
                        file_id=file_id,
                        file_name=file_name,
                        target_dir=Path.cwd(),
                    )
                    print(f"Saved image file to: {Path.cwd() / file_name}")
                    download_succeeded = True
                    found_file = True
                    break
                except Exception as ex:
                    wait_seconds = attempt * 2
                    print(f"Attempt {attempt}/6 failed to download file {file_id}: {ex}")
                    time.sleep(wait_seconds)

            if not download_succeeded:
                print(f"Chart was generated by the agent, but download failed for file id: {file_id}")

        if not found_file:
            print("Agent generated output, but file download failed due to a transient service issue.")

    finally:
        if thread is not None:
            try:
                project.agents.threads.delete(thread_id=thread.id)
                print(f"Deleted thread: {thread.id}")
            except Exception as ex:
                print(f"Failed to delete thread {thread.id}: {ex}")

        try:
            project.agents.delete_agent(agent.id)
            print("Deleted agent")
        except Exception as ex:
            print(f"Failed to delete agent {agent.id}: {ex}")

Created agent, ID: asst_3OM6Q7O8SzRNYfiph4LxDfeq
Created thread, ID: thread_G8nunVZpzf8wAeiv44BnnuIn
Created message, ID: msg_AyjQpAVw3KNGv6KIzyg42lX3
Run finished with status: RunStatus.COMPLETED
Conversation:
MessageRole.AGENT: [{'type': 'text', 'text': {'value': 'The bar chart comparing the health plans has been created. You can download the file using the link below:\n\n[Download health-plan-comparision.png](sandbox:/mnt/data/health-plan-comparision.png)', 'annotations': [{'type': 'file_path', 'text': 'sandbox:/mnt/data/health-plan-comparision.png', 'start_index': 147, 'end_index': 192, 'file_path': {'file_id': 'assistant-UXXyeeSu1yN2FC4FvCjoii'}}]}}]
Attempt 1/6 failed to download file assistant-UXXyeeSu1yN2FC4FvCjoii: (None) The server had an error processing your request. Sorry about that! You can retry your request, or contact us through our help center at oai-assistants@microsoft.com if you keep seeing this error. (Please include the request ID 898aae15-4c58-45ab-ae07-1cb11b80